<a href="https://colab.research.google.com/github/Zeneck-W/ie332-fall2026/blob/main/SQL_Lec2_First_Queries_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IE 332 - SQL Lectures 1 and 2: Meet the Data, then First Queries

**Companion Colab notebook**

**Part 1, meet the data:** what is actually inside this database, and what does one row of each table mean?

**Part 2, first queries:** how do we start with a whole table and then shape the answer one change at a time?

- `SELECT` chooses columns or expressions.
- `FROM` names the source table.
- `WHERE` filters rows before they reach the result.
- `ORDER BY` sorts; `LIMIT` controls how many rows are displayed.

Keep `boilermaker_brews.db` beside this notebook, or upload it when the setup cell asks. Executed outputs are saved, so you can read the notebook even without running it.


## Setup

Run this once. It connects to the campus-cafe sales SQLite database and defines a lightweight `%%sql` command. The setup is Python; every demonstration after it is pure SQL.

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
from IPython.display import display

# show a compact view of any result: first and last rows, plus its shape
pd.set_option("display.max_rows", 10)
pd.set_option("display.width", 120)

DB_PATH = Path("boilermaker_brews.db")

if not DB_PATH.exists():
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(
            "Place boilermaker_brews.db in the notebook's working folder."
        ) from exc

    print("Choose boilermaker_brews.db from the course files.")
    files.upload()

if not DB_PATH.exists():
    raise FileNotFoundError("boilermaker_brews.db was not uploaded.")

conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")

def _execute_sql(connection, statement):
    statement = statement.strip()
    first_word = statement.split(None, 1)[0].upper()
    if first_word in {"SELECT", "WITH", "PRAGMA", "EXPLAIN"}:
        result = pd.read_sql_query(statement, connection)
        display(result)
        return

    connection.executescript(statement)
    connection.commit()
    print("Statement executed successfully.")

def _sql_magic(line, cell):
    _execute_sql(conn, cell)

def _expected_error_magic(line, cell):
    try:
        _execute_sql(conn, cell)
    except Exception as error:
        print(f"Expected error: {type(error).__name__}: {error}")
    else:
        raise AssertionError("This demonstration was expected to produce an SQL error.")

ip = get_ipython()
ip.register_magic_function(_sql_magic, "cell", "sql")
ip.register_magic_function(_expected_error_magic, "cell", "sql_expect_error")

table_count = pd.read_sql_query(
    "SELECT COUNT(*) AS n FROM sqlite_master WHERE type='table';", conn
).iloc[0, 0]
print(f"Connected to {DB_PATH.name}: {table_count} tables. The %%sql demo command is ready.")

Connected to boilermaker_brews.db: 6 tables. The %%sql demo command is ready.


# Part 1: meet the data

Six tables. Open every one of them with `SELECT *`, the query that asks for all columns.


## Which tables exist?

Every SQLite database keeps a catalog of itself in `sqlite_master`.


In [ ]:
%%sql
SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name;


,name
0,customers
1,employees
2,order_items
3,orders
4,products
5,stores


## How big is each table?

One row per table, so the whole database fits on one screen.


In [ ]:
%%sql
SELECT 'stores' AS table_name, COUNT(*) AS n_rows FROM stores
UNION ALL SELECT 'products',    COUNT(*) FROM products
UNION ALL SELECT 'customers',   COUNT(*) FROM customers
UNION ALL SELECT 'employees',   COUNT(*) FROM employees
UNION ALL SELECT 'orders',      COUNT(*) FROM orders
UNION ALL SELECT 'order_items', COUNT(*) FROM order_items;


,table_name,n_rows
0,stores,6
1,products,26
2,customers,600
3,employees,36
4,orders,51927
5,order_items,78777


**Read the result:** four small reference tables and two large ones. The big tables record events (checkouts and the products on them); the small ones describe the things those events point at.


## `stores`

Every location the chain operates.


In [ ]:
%%sql
SELECT * FROM stores;


,store_id,name,campus_area,opened_date,seats
0,1,Chauncey Hill,Chauncey,2019-08-12,38
1,2,PMU Ground Floor,Memorial Union,2017-01-09,64
2,3,Discovery Park,Discovery Park,2022-03-21,22
3,4,Levee Plaza,Levee,2020-09-01,30
4,5,State Street East,State Street,2018-05-14,26
5,6,Airport Rd Drive-Thru,South Campus,2023-10-02,0


**Read the result:** One row = one cafe. `store_id` identifies it; the other columns describe it.


## `products`

The full menu.


In [ ]:
%%sql
SELECT * FROM products;


,product_id,name,category,price,is_seasonal
0,1,Drip Coffee 12oz,brew,2.50,0
1,2,Drip Coffee 16oz,brew,3.00,0
2,3,Cold Brew 16oz,brew,4.25,0
3,4,Nitro Cold Brew,brew,4.95,0
4,5,Espresso (double),espresso,3.25,0
...,...,...,...,...,...
21,22,Chocolate Chip Cookie,pastry,2.50,0
22,23,Turkey Pesto Sandwich,food,7.50,0
23,24,Caprese Sandwich,food,7.25,0
24,25,BB Travel Mug,merch,18.00,0


**Read the result:** One row = one menu item. Note `is_seasonal`: SQLite has no true/false type, so it is stored as 1/0.


## `employees`

Everyone who works the counter.


In [ ]:
%%sql
SELECT * FROM employees;


,employee_id,name,store_id,hired_date,hourly_wage
0,1,Tyler Park,1,2023-06-07,11.25
1,2,Ethan Clark,1,2026-02-22,15.82
2,3,Hana Davis,1,2024-02-04,13.79
3,4,Noah Shah,1,2026-03-03,14.77
4,5,Priya Johnson,1,2023-12-31,11.78
...,...,...,...,...,...
31,32,Mateo Adams,6,2021-02-15,14.19
32,33,Wei Baker,6,2026-05-24,12.19
33,34,Wei Chen,6,2021-05-13,15.10
34,35,Ethan Martinez,6,2022-12-29,16.00


**Read the result:** One row = one employee. `store_id` is not a fact about the person; it points at a row of `stores`.


## `customers`

Loyalty-program members only.


In [ ]:
%%sql
SELECT * FROM customers;


,customer_id,name,joined_date,loyalty_tier
0,1,Gina Johnson,2025-01-16,silver
1,2,Nora Lee,2025-02-22,none
2,3,Zoe Cruz,2024-05-16,silver
3,4,Mia Hall,2025-09-15,gold
4,5,Lena Patel,2024-07-16,none
...,...,...,...,...
595,596,Ravi Cruz,2025-05-12,none
596,597,Wei Singh,2025-11-15,none
597,598,Chen Lee,2024-10-14,silver
598,599,Jordan Baker,2026-01-19,none


**Read the result:** One row = one member. 600 rows, but the chain serves far more people than that: walk-ins are not members.


## `orders`

One year of checkouts.


In [ ]:
%%sql
SELECT * FROM orders;


,order_id,store_id,customer_id,employee_id,order_ts,channel
0,1,1,260.0,1,2025-09-01 09:03:31,counter
1,2,1,NaN,2,2025-09-01 10:58:37,counter
2,3,1,400.0,4,2025-09-01 07:08:41,counter
3,4,1,179.0,2,2025-09-01 16:28:49,counter
4,5,1,527.0,3,2025-09-01 18:38:47,counter
...,...,...,...,...,...,...
51922,51923,6,560.0,36,2026-08-31 10:43:13,counter
51923,51924,6,NaN,31,2026-08-31 12:04:20,counter
51924,51925,6,NaN,32,2026-08-31 10:28:22,counter
51925,51926,6,438.0,32,2026-08-31 11:36:27,app


**Read the result:** One row = one checkout. Most columns are other tables' IDs: which store, which customer, which employee. And some `customer_id` values are missing, the walk-ins.

Two display details worth knowing now: Python prints SQL's `NULL` as `NaN`, and a column holding any missing value is shown with decimals (`260.0`), even though the database stores plain integers and `NULL`. The storage is unchanged; only the on-screen rendering differs.


## `order_items`

What was actually in each order.


In [ ]:
%%sql
SELECT * FROM order_items;


,order_id,product_id,quantity,unit_price
0,1,10,1,5.25
1,2,16,1,2.75
2,2,6,1,3.50
3,2,7,1,4.25
4,3,12,1,5.75
...,...,...,...,...
78772,51924,14,1,5.00
78773,51925,17,1,2.75
78774,51925,10,1,5.25
78775,51926,14,1,5.00


**Read the result:** One row = one product on one order, so an order with three drinks appears three times. `unit_price` is a copy of the price at the moment of sale.


## Reading a table you have never seen

The same three questions work on any table, in any database:

1. **What is one row?** State it in one sentence. This is the table's *grain*.
2. **Which column identifies a row?** That is its key.
3. **Which columns point somewhere else?** Those IDs are the links between tables.

Answer them before writing a query, and most query bugs never happen.


# Part 2: first queries

Now shape the answer: choose columns, filter rows, sort, and limit.


## Types on real data

`typeof()` reports each stored value's storage class. Then: the floating-point trap.


In [ ]:
%%sql
SELECT price, typeof(price), is_seasonal, typeof(is_seasonal)
FROM products
LIMIT 3;


,price,typeof(price),is_seasonal,typeof(is_seasonal)
0,2.50,real,0,integer
1,3.00,real,0,integer
2,4.25,real,0,integer


**Read the result:** `price` is stored as `real`, and `is_seasonal` as `integer`. SQLite has no Boolean storage class, so true/false lives as `1`/`0`.


In [ ]:
%%sql
SELECT order_id, order_ts, typeof(order_ts), customer_id, typeof(customer_id)
FROM orders
LIMIT 3;


,order_id,order_ts,typeof(order_ts),customer_id,typeof(customer_id)
0,1,2025-09-01 09:03:31,text,260.0,integer
1,2,2025-09-01 10:58:37,text,NaN,null
2,3,2025-09-01 07:08:41,text,400.0,integer


**Read the result:** timestamps are stored as `text` (SQLite has no date type). And order 2's walk-in `customer_id` reports `null`: NULL is its own storage class. Hold that thought for Step 16.


In [ ]:
%%sql
SELECT name, price, typeof(price)
FROM products
WHERE typeof(price) != 'real';


,name,price,typeof(price)


**Read the result:** zero rows, and here zero rows is the good answer: every stored `price` really is `real`. This one-line audit is how you check a column's types instead of assuming them. SQLite does not enforce declared types, so on messy imported data this same query returns the offending rows.


### Trap 1: floating-point


In [ ]:
%%sql
SELECT 0.1 + 0.2 = 0.3 AS equals_point_three,
       (0.1 + 0.2) - 0.3 AS difference;


,equals_point_three,difference
0,0,5.551115e-17


**Read the result:** `equals_point_three` is `0`, that is, false. The difference is about `5.6e-17`: tiny, but not zero, so `=` fails.


In [ ]:
# Same IEEE arithmetic in Python: here are the digits in full.
0.1 + 0.2


0.30000000000000004

**Read the result:** `0.30000000000000004`, the number on the slide. Floating-point cannot represent 0.1 or 0.2 exactly, so their sum is not exactly 0.3. The fix: store money as whole cents in an `INTEGER` column; convert to dollars only for the final display.


## A. Begin with the whole table, then choose what you need

### Step 1 - Show every column

**Starting point:** Ask for every column from `products`, but display only five rows so the result is easy to inspect.

**Predict before running:** What does one output row represent? How many columns will appear?

In [ ]:
%%sql
SELECT *
FROM   products
LIMIT  5;

,product_id,name,category,price,is_seasonal
0,1,Drip Coffee 12oz,brew,2.50,0
1,2,Drip Coffee 16oz,brew,3.00,0
2,3,Cold Brew 16oz,brew,4.25,0
3,4,Nitro Cold Brew,brew,4.95,0
4,5,Espresso (double),espresso,3.25,0


**Read the result:** The result has five rows and all five product columns. One row still means one product. `LIMIT` changes only how many rows we see; it does not change the table itself.

### Step 2 - Keep only useful columns

**Only change:** Replace `*` with `name, price`. The table and the five-row limit stay unchanged.

**Predict before running:** Which columns disappear? Should the number of displayed rows change?

In [ ]:
%%sql
SELECT name, price
FROM   products
LIMIT  5;

,name,price
0,Drip Coffee 12oz,2.50
1,Drip Coffee 16oz,3.00
2,Cold Brew 16oz,4.25
3,Nitro Cold Brew,4.95
4,Espresso (double),3.25


**Read the result:** The same five products remain, but the result now has only two columns. `SELECT` shapes columns; it does not filter rows.

### Step 3 - Add a calculated column

**Only change:** Keep `name` and add the expression `ROUND(price * 1.07, 2)`. Give that output column a readable alias with `AS`.

**Predict before running:** Does this calculation modify the stored `price`, or only the returned result?

In [ ]:
%%sql
SELECT name,
       ROUND(price * 1.07, 2) AS price_with_tax
FROM   products
LIMIT  5;

,name,price_with_tax
0,Drip Coffee 12oz,2.68
1,Drip Coffee 16oz,3.21
2,Cold Brew 16oz,4.55
3,Nitro Cold Brew,5.30
4,Espresso (double),3.48


**Read the result:** SQL calculates `price_with_tax` for each returned row. The source table is unchanged; the alias exists only in this result.

## B. Remove duplicate values

### Step 4 - First look at the raw category column

**Starting point:** Start with one ordinary column and display ten rows.

**Predict before running:** Will category names repeat? Why?

In [ ]:
%%sql
SELECT category
FROM   products
ORDER  BY category
LIMIT  10;

,category
0,brew
1,brew
2,brew
3,brew
4,espresso
5,espresso
6,espresso
7,espresso
8,espresso
9,espresso


**Read the result:** Categories repeat because several products belong to the same category. One output row still corresponds to one source product.

### Step 5 - Add `DISTINCT`

**Only change:** Insert only the word `DISTINCT` after `SELECT` and remove the display limit so every unique category can appear.

**Predict before running:** Will the result still have 26 product-level rows, or one row per different category?

In [ ]:
%%sql
SELECT DISTINCT category
FROM   products
ORDER  BY category;

,category
0,brew
1,espresso
2,food
3,merch
4,pastry
5,tea


**Read the result:** The result collapses repeated category values into six unique rows. `DISTINCT` answers 'which values occur?', not 'how often?'.

### Step 5b - `DISTINCT` on two columns

Predict first: `DISTINCT category` returned six rows. How many rows does `DISTINCT category, is_seasonal` return?


In [ ]:
%%sql
SELECT DISTINCT category, is_seasonal
FROM products
ORDER BY category, is_seasonal;


,category,is_seasonal
0,brew,0
1,espresso,0
2,espresso,1
3,food,0
4,merch,0
5,pastry,0
6,tea,0


**Read the result:** seven rows, not six and not twelve. `DISTINCT` keeps each distinct *combination* of the selected columns, and espresso is the only category containing both a seasonal and a non-seasonal item, so it appears twice. The unit is the whole selected row; there is no per-column `DISTINCT`.


## C. Keep rows that satisfy a condition

### Step 6 - Reset to a simple product list

**Starting point:** Return three descriptive columns from the product table.

**Predict before running:** Before filtering, what does one result row mean?

In [ ]:
%%sql
SELECT name, category, price
FROM   products
ORDER  BY product_id;

,name,category,price
0,Drip Coffee 12oz,brew,2.50
1,Drip Coffee 16oz,brew,3.00
2,Cold Brew 16oz,brew,4.25
3,Nitro Cold Brew,brew,4.95
4,Espresso (double),espresso,3.25
...,...,...,...
21,Chocolate Chip Cookie,pastry,2.50
22,Turkey Pesto Sandwich,food,7.50
23,Caprese Sandwich,food,7.25
24,BB Travel Mug,merch,18.00


**Read the result:** All 26 products appear. This is the baseline result that the next conditions will filter.

### Step 7 - Add one `WHERE` test

**Only change:** Add `WHERE price < 3.00`; keep the selected columns and table unchanged.

**Predict before running:** Which rows should disappear? Will any columns disappear?

In [ ]:
%%sql
SELECT name, category, price
FROM   products
WHERE  price < 3.00
ORDER  BY product_id;

,name,category,price
0,Drip Coffee 12oz,brew,2.50
1,Earl Grey Tea,tea,2.75
2,Green Tea,tea,2.75
3,Everything Bagel,pastry,2.95
4,Chocolate Chip Cookie,pastry,2.50


**Read the result:** Only five inexpensive products remain, but all three selected columns remain. `WHERE` changes rows, not columns.

### Step 8 - Change the test from numeric to text

**Only change:** Replace the price comparison with `category = 'espresso'`. Strings use single quotes.

**Predict before running:** How many espresso products do you expect?

In [ ]:
%%sql
SELECT name, category, price
FROM   products
WHERE  category = 'espresso'
ORDER  BY product_id;

,name,category,price
0,Espresso (double),espresso,3.25
1,Americano,espresso,3.50
2,Cappuccino,espresso,4.25
3,Latte 12oz,espresso,4.50
4,Latte 16oz,espresso,5.00
5,Caramel Latte,espresso,5.25
6,Mocha,espresso,5.25
7,Pumpkin Spice Latte,espresso,5.75
8,Peppermint Mocha,espresso,5.75


**Read the result:** The query now keeps all products whose category text exactly equals `espresso`. The query shape is unchanged; only the condition changed.

### Step 9 - Require two tests with `AND`

**Only change:** Add one second condition: `price <= 4.50`.

**Predict before running:** Should the result become larger or smaller when both tests must be true?

In [ ]:
%%sql
SELECT name, category, price
FROM   products
WHERE  category = 'espresso'
  AND  price <= 4.50
ORDER  BY product_id;

,name,category,price
0,Espresso (double),espresso,3.25
1,Americano,espresso,3.50
2,Cappuccino,espresso,4.25
3,Latte 12oz,espresso,4.50


**Read the result:** The result becomes smaller because a row must pass both tests. `AND` narrows a result.

### Step 10 - Use `OR` for either category

**Only change:** Replace the previous condition with two category tests joined by `OR`.

**Predict before running:** Should a tea pass? Should a brew pass? Should an espresso pass?

In [ ]:
%%sql
SELECT name, category, price
FROM   products
WHERE  category = 'tea' OR category = 'brew'
ORDER  BY category, name;

,name,category,price
0,Cold Brew 16oz,brew,4.25
1,Drip Coffee 12oz,brew,2.50
2,Drip Coffee 16oz,brew,3.00
3,Nitro Cold Brew,brew,4.95
4,Chai Latte,tea,4.75
5,Earl Grey Tea,tea,2.75
6,Green Tea,tea,2.75
7,Iced Black Tea,tea,3.25
8,Matcha Latte,tea,5.00


**Read the result:** A row survives when either comparison is true. `OR` broadens the set relative to either condition by itself.

### Step 11 - Rewrite repeated `OR` tests with `IN`

**Only change:** Replace the two equality comparisons with `category IN ('tea', 'brew')`.

**Predict before running:** Should the rows change, or only the way the condition is written?

In [ ]:
%%sql
SELECT name, category, price
FROM   products
WHERE  category IN ('tea', 'brew')
ORDER  BY category, name;

,name,category,price
0,Cold Brew 16oz,brew,4.25
1,Drip Coffee 12oz,brew,2.50
2,Drip Coffee 16oz,brew,3.00
3,Nitro Cold Brew,brew,4.95
4,Chai Latte,tea,4.75
5,Earl Grey Tea,tea,2.75
6,Green Tea,tea,2.75
7,Iced Black Tea,tea,3.25
8,Matcha Latte,tea,5.00


**Read the result:** The result is identical to the previous query. `IN` is the clearer form when one column may match any value in a list.

### Step 12 - Replace the condition with an inclusive range

**Only change:** Use `price BETWEEN 3.00 AND 5.00` while keeping the rest of the query unchanged.

**Predict before running:** Will products priced exactly $3.00 or $5.00 be included?

In [ ]:
%%sql
SELECT name, category, price
FROM   products
WHERE  price BETWEEN 3.00 AND 5.00
ORDER  BY price, name;

,name,category,price
0,Drip Coffee 16oz,brew,3.00
1,Blueberry Muffin,pastry,3.25
2,Espresso (double),espresso,3.25
3,Iced Black Tea,tea,3.25
4,Americano,espresso,3.50
...,...,...,...
9,Latte 12oz,espresso,4.50
10,Chai Latte,tea,4.75
11,Nitro Cold Brew,brew,4.95
12,Latte 16oz,espresso,5.00


**Read the result:** Both endpoints appear because `BETWEEN` is inclusive. It is equivalent to `price >= 3.00 AND price <= 5.00`.

### Step 13 - Replace the condition with a text pattern

**Only change:** Use `name LIKE '%Latte%'`. The percent signs allow any text before or after `Latte`.

**Predict before running:** Would the exact test `name = 'Latte'` find these same products?

In [ ]:
%%sql
SELECT name, category, price
FROM   products
WHERE  name LIKE '%Latte%'
ORDER  BY name;

,name,category,price
0,Caramel Latte,espresso,5.25
1,Chai Latte,tea,4.75
2,Latte 12oz,espresso,4.50
3,Latte 16oz,espresso,5.00
4,Matcha Latte,tea,5.00
5,Pumpkin Spice Latte,espresso,5.75


**Read the result:** Six names contain the pattern. `LIKE` performs pattern matching; `=` requires an exact value.

### Step 14 - Group mixed `AND` and `OR` logic

**Only change:** Start with the tea-or-brew condition, surround it with parentheses, and add `AND price < 4.00`.

**Predict before running:** Which test should be evaluated as one group? Why do the parentheses matter?

In [ ]:
%%sql
SELECT name, category, price
FROM   products
WHERE  (category = 'tea' OR category = 'brew')
  AND  price < 4.00
ORDER  BY category, name;

,name,category,price
0,Drip Coffee 12oz,brew,2.50
1,Drip Coffee 16oz,brew,3.00
2,Earl Grey Tea,tea,2.75
3,Green Tea,tea,2.75
4,Iced Black Tea,tea,3.25


**Read the result:** A row must be tea or brew, and it must also cost under $4. Parentheses make the intended logic explicit.

## D. Handle missing values correctly

### Step 15 - Look at a nullable column

**Starting point:** Switch to `orders` and inspect `customer_id`. Walk-in orders have no customer account, so this column may be missing.

**Predict before running:** How does SQLite display a missing customer ID?

In [ ]:
%%sql
SELECT order_id, customer_id
FROM   orders
LIMIT  10;

,order_id,customer_id
0,1,260.0
1,2,NaN
2,3,400.0
3,4,179.0
4,5,527.0
5,6,374.0
6,7,NaN
7,8,318.0
8,9,106.0
9,10,167.0


**Read the result:** Missing database values appear as `None` in this Colab table; SQL calls the value `NULL`.

### Step 16 - Try ordinary equality with `NULL`

**Only change:** Add `WHERE customer_id = NULL`.

**Predict before running:** Will SQL treat two unknown values as equal?

In [ ]:
%%sql
SELECT order_id, customer_id
FROM   orders
WHERE  customer_id = NULL
LIMIT  10;

,order_id,customer_id


**Read the result:** No rows pass. A comparison with unknown is itself unknown, not true, so `= NULL` is the wrong test.

### Step 17 - Change only the `NULL` operator

**Only change:** Replace `= NULL` with `IS NULL`.

**Predict before running:** Should walk-in orders now appear?

In [ ]:
%%sql
SELECT order_id, customer_id
FROM   orders
WHERE  customer_id IS NULL
LIMIT  10;

,order_id,customer_id
0,2,None
1,7,None
2,14,None
3,19,None
4,22,None
5,24,None
6,29,None
7,32,None
8,34,None
9,44,None


**Read the result:** Walk-in orders now appear. Use `IS NULL` and `IS NOT NULL` for missing-value tests.

### Step 18 - Change the projection to a count

**Only change:** Replace the two displayed columns with `COUNT(*) AS walk_in_orders`; keep the correct filter.

**Predict before running:** Will the result contain thousands of rows or one summary row?

In [ ]:
%%sql
SELECT COUNT(*) AS walk_in_orders
FROM   orders
WHERE  customer_id IS NULL;

,walk_in_orders
0,18203


**Read the result:** The result is one row containing 18,203, the number of orders that passed the filter.

## E. Turn a preview into a top-N answer

### Step 19 - Begin with three arbitrary displayed rows

**Starting point:** Return product names and prices, then limit the display to three rows.

**Predict before running:** Without sorting, do these rows mean the three most expensive products?

In [ ]:
%%sql
SELECT name, price
FROM   products
LIMIT  3;

,name,price
0,Drip Coffee 12oz,2.50
1,Drip Coffee 16oz,3.00
2,Cold Brew 16oz,4.25


**Read the result:** These are merely three rows from the table's current scan order. Without `ORDER BY`, 'first' does not mean highest or lowest.

### Step 20 - Add a descending sort before the limit

**Only change:** Insert `ORDER BY price DESC` before `LIMIT 3`.

**Predict before running:** Which product should move to the first row?

In [ ]:
%%sql
SELECT name, price
FROM   products
ORDER  BY price DESC
LIMIT  3;

,name,price
0,BB Travel Mug,18.00
1,Turkey Pesto Sandwich,7.50
2,Caprese Sandwich,7.25


**Read the result:** Sorting first and limiting second produces the three highest-priced products. `DESC` means largest to smallest; `ASC` is the default.

## Details and common questions

### Detail 1 - Is SQLite case-sensitive?

There is no single yes/no answer. For the operations used here, default `LIKE` treats ASCII uppercase and lowercase letters as equivalent; ordinary `=` on the course text columns uses the case-sensitive `BINARY` collation; and `GLOB` is case-sensitive. The underscore wildcard matches exactly one character, while `%` matches zero or more characters. Predict all four `0`/`1` results before running.


In [ ]:
%%sql
SELECT 'Latte' LIKE 'latte' AS like_ignores_ascii_case,
       'tea' = 'Tea' AS default_equality_is_case_sensitive,
       'Latte' LIKE 'L_tt_' AS underscore_matches_one,
       'Latte' GLOB 'latte' AS glob_is_case_sensitive;


,like_ignores_ascii_case,default_equality_is_case_sensitive,underscore_matches_one,glob_is_case_sensitive
0,1,0,1,0


**Read the result:** The values are `1`, `0`, `1`, and `0`. SQLite's default case folding for `LIKE` covers ASCII letters, not every Unicode alphabet. Ordinary `=` follows the column's collation; the course text columns use the default case-sensitive `BINARY` collation. `GLOB` is case-sensitive and uses `*` and `?`, not `%` and `_`. Pattern-matching defaults vary across database systems, so identify the SQL dialect before depending on case behavior.

### Detail 2 - How are mixed `AND` and `OR` conditions grouped?

After comparisons, SQLite's Boolean precedence is `NOT`, then `AND`, then `OR`. Before running the next query, choose its equivalent meaning:

- **A:** `(category = 'tea' OR category = 'brew') AND price < 3.00`
- **B:** `category = 'tea' OR (category = 'brew' AND price < 3.00)`

Then predict which products survive.


In [ ]:
%%sql
SELECT name, category, price
FROM   products
WHERE  category = 'tea'
   OR  category = 'brew' AND price < 3.00
ORDER  BY category, price, name;


,name,category,price
0,Drip Coffee 12oz,brew,2.50
1,Earl Grey Tea,tea,2.75
2,Green Tea,tea,2.75
3,Iced Black Tea,tea,3.25
4,Chai Latte,tea,4.75
5,Matcha Latte,tea,5.00


**Read the result:** Every tea survives, regardless of price, but only brews below $3 survive. Parentheses can override precedence. They also make the intention visible, so use them whenever `AND` and `OR` are mixed.

Precedence describes logical grouping; it does not guarantee that the database physically evaluates predicates from left to right.


## The pattern to remember

```sql
SELECT   columns_or_expressions
FROM     table
WHERE    row_condition
ORDER BY sort_expression [ASC | DESC]
LIMIT    number_of_rows;
```

Read every result as a table: state what one row means, which columns appear, and why each row survived. Strings use single quotes. `DISTINCT` applies to the complete selected row. `BETWEEN` includes both endpoints. In SQLite, `LIKE` ignores ASCII case, `%` matches any-length text, and `_` matches one character. Boolean precedence is `NOT` > `AND` > `OR`; parenthesize mixed logic. Use `IS NULL`, never `= NULL`. Never call a limited result "first" or "top" without `ORDER BY`.
